# Exercise 10: Capstone — The Complete Genomics Data Platform
## BINFX410 — Chapter 10: Data Lakes, Warehouses, and Lakehouses

---

## Purpose of This Notebook

This capstone notebook ties together all nine exercises into a single coherent architecture. By the time you finish reading it, you should be able to:

- Draw the full data platform from raw sequencer output to governed clinical analytics
- Explain which AWS service belongs at each stage and why
- Identify the key tradeoffs (cost vs. latency, flexibility vs. governance, batch vs. streaming)
- Size and estimate the monthly cost of a realistic genomics data lake
- Connect the platform to real-world bioinformatics standards (FAIR, GA4GH, dbGaP)

**This notebook contains no executable code.** All code blocks are shown for reference and discussion only.

**Estimated reading time:** 60 minutes

## The Journey: Nine Exercises, One Platform

Each exercise addressed a specific layer or capability of a production genomics data platform. Here is where each one fits:

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                  The BINFX410 Genomics Data Platform                         │
│                                                                              │
│  DATA SOURCES            INGEST              STORAGE (S3)                   │
│  ────────────            ──────              ────────────                   │
│  Sequencer runs  ──►  [Ex 9: Kinesis]  ──►  BRONZE  (raw)                  │
│  Public DBs      ──►  [Ex 1: S3 copy]  ──►  ├─ variants/                   │
│  Clinical EHR    ──►  [Ex 9: Firehose]      ├─ rna-seq/                    │
│                                              └─ streaming/                  │
│                                                   │                         │
│  CATALOG & FORMAT       TRANSFORM                 ▼                         │
│  ────────────────        ─────────             SILVER  (clean)              │
│  [Ex 2: Parquet]  ◄──  [Ex 4: Glue ELT] ──►  ├─ variants.parquet          │
│  [Ex 3: Glue Cat]        PySpark jobs          └─ counts.parquet            │
│                                                   │                         │
│  ANALYTICAL LAYER        MODELING                 ▼                         │
│  ────────────────        ────────              GOLD  (aggregated)           │
│  [Ex 5: Star schema]◄──[Ex 4: CTAS]   ──►   ├─ fact_sequencing_run        │
│  [Ex 6: Iceberg]                              ├─ dim_sample                 │
│  [Ex 7: Partitions]                           └─ variant_summary            │
│                                                                              │
│  GOVERNANCE              QUERY ENGINE                                        │
│  ──────────              ────────────                                        │
│  [Ex 8: Lake Form.]      Amazon Athena  ◄── all layers queryable            │
│  IAM roles               dbt models     ◄── [Ex 5]                         │
│  Column/row filters                                                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

## Exercise-by-Exercise Summary

### Exercise 1 — Genomics Data Lake Anatomy
**Core question answered:** Where does genomics data live, and how do we organize it?

You created the foundational three-zone (Bronze/Silver/Gold) S3 bucket structure and staged real GATK test data into the Bronze zone using Hive-style partitioning (`sample=NA12878/reference=hg38/`). The key insight was **schema-on-read**: the same S3 object can be read as raw bytes, a pandas DataFrame, or a structured table depending on the tool — no ETL required to read it.

**What you built:** S3 bucket with zone structure, Glue database, Athena table over CSV data, partition pruning demo.

**Key decision made:** Organize by zone first, then by domain, then by Hive partition keys. Never delete Bronze.

---

### Exercise 2 — File Format Showdown
**Core question answered:** Does format choice really matter at genomics scale?

You downloaded 50,000 ClinVar variants, stored them as both VCF-style TSV and Parquet, and benchmarked Athena query cost against both. Parquet's columnar storage meant a query selecting 3 of 43 columns scanned **~900× less data** than the equivalent VCF query — translating to $0.02/month vs. $14.65/month for a realistic query workload.

**What you built:** Parquet vs. TSV benchmark tables, cost visualization, format conversion pipeline.

**Key decision made:** Always convert to Parquet at the Silver layer. Keep original format in Bronze. For very wide tables (VCF with hundreds of INFO fields), column projection pays exponentially more.

---

### Exercise 3 — Glue Data Catalog
**Core question answered:** How do teams discover and trust data across a growing lake?

You staged synthetic multi-omics TCGA data (mutations, RNA-seq, methylation) and automated schema discovery using Glue crawlers. Crawlers infer column types, partition keys, and row counts, writing metadata to the Glue Data Catalog — the shared discovery layer that Athena, Glue ETL, and Lake Formation all query.

**What you built:** Glue crawler, multi-table database, schema versioning demo.

**Key decision made:** Use crawlers for data you don't control (external files, new uploads). Use `awswrangler.s3.to_parquet(dataset=True)` for data you write yourself — it registers the table without a crawler.

---

### Exercise 4 — ELT Pipeline for RNA-seq
**Core question answered:** How do you transform messy raw count data into analysis-ready tables at scale?

You wrote a three-stage Glue PySpark job that moves RNA-seq data through the medallion layers: Bronze (raw count matrix) → Silver (filtered, TMM-normalized, deduplicated Parquet) → Gold (per-gene summary statistics with highly variable gene flags). This is the ELT pattern: **Extract** from Bronze, **Load** to Silver/Gold as Parquet, **Transform** in-place using Spark.

**What you built:** Glue PySpark ETL script, three-layer normalized RNA-seq tables, Athena-queryable Gold layer.

**Key decision made:** Normalize counts in the Silver layer, not at query time. Athena is not designed for row-by-row math across millions of rows; Spark is. Do heavy computation once in ETL, query the results many times cheaply.

---

### Exercise 5 — Star Schema with SRA Metadata
**Core question answered:** How do you model complex metadata relationships for efficient analytical queries?

You generated synthetic SRA run metadata and decomposed it into a star schema: four dimension tables (`dim_platform`, `dim_sample`, `dim_study`, `dim_date`) and one fact table (`fact_sequencing_run`) using Athena CTAS. You also wrote dbt models using `{{ ref() }}` to declare cross-table dependencies, enabling automated dependency-ordered rebuilds.

**What you built:** Star schema in Athena, three analytical queries (platform × organism, year-over-year growth, disease × tissue), dbt model SQL files.

**Key decision made:** Star schemas are not just for data warehouses — they work equally well in a lakehouse when tables are Parquet on S3. The `ref()` pattern in dbt is more important than the format: it makes the lineage graph explicit and reproducible.

---

### Exercise 6 — Apache Iceberg and Time Travel
**Core question answered:** How do you handle variant reclassification, schema changes, and audit requirements without breaking downstream queries?

You created an Iceberg table of ClinVar variants and simulated a clinical reclassification workflow: INSERT, UPDATE (VUS → Pathogenic), ADD COLUMN, DELETE. Each operation created a new immutable snapshot. Time travel queries (`FOR TIMESTAMP AS OF`) let you reconstruct the exact table state at any point in history — a requirement for HIPAA audit trails and for reproducible research.

**What you built:** Iceberg table, ACID DML operations via Athena v3, snapshot history inspection, time travel queries.

**Key decision made:** Use Iceberg for any table that requires UPDATE/DELETE, schema evolution, or regulatory audit trails. Use plain Parquet (append-only) for tables that never change (Bronze staging, raw counts). Iceberg's metadata overhead is not worth it for write-once data.

---

### Exercise 7 — Partitioning Strategies
**Core question answered:** How many partition keys is too many, and which key delivers the most query savings?

You benchmarked three partitioning strategies for 1000 Genomes chr22 data against three query patterns. The results were clear: chromosome partitioning reduced scanned bytes by 50–60× for chromosome-filtered queries. Over-partitioning (chrom + superpopulation + variant_type) created 30+ tiny files and **increased** cost on full-table scans due to per-file overhead.

**What you built:** Three Parquet datasets with different partition strategies, benchmark query results, file statistics analysis.

**Key decision made:** Partition on the column(s) that appear most frequently in your WHERE clauses. Aim for partitions containing 128 MB–1 GB of data each. If you have too many small files, compact them with an Iceberg `OPTIMIZE` or a Glue compaction job before they accumulate.

---

### Exercise 8 — Lake Formation Governance
**Core question answered:** How do you enforce HIPAA minimum-necessary access across researchers with different clearance levels?

You generated 50 synthetic patient records containing PHI (MRN, DOB, SSN-last4) alongside genomic results, then used Lake Formation to grant two IAM roles different access: the clinical role saw all columns and all 50 rows; the researcher role saw only non-PHI columns and only the Brain tissue rows (via data cell filter). You verified access using `sts:AssumeRole`.

**What you built:** Synthetic PHI dataset, IAM roles with trust policies, Lake Formation column permissions, row-level data cell filter, access verification via role assumption.

**Key decision made:** Lake Formation governs access at the table/column/row level, but the enforcement point is the query engine (Athena). S3 bucket policies alone cannot enforce column masking — they only control bucket/prefix access. LF + Athena together enable the HIPAA minimum-necessary principle at query time.

---

### Exercise 9 — Kinesis Streaming
**Core question answered:** How do you ingest continuous event streams from sequencers or clinical systems in near real time?

You worked through all three Kinesis services: Data Streams (low-latency pub/sub with shard-based ordering), Firehose (managed delivery to S3 bronze with buffering and GZIP compression), and Data Analytics (real-time SQL for windowed aggregations and clinical alerting). You connected the streaming bronze layer to the same medallion architecture used in the batch exercises.

**What you built:** Kinesis Data Stream with partition-key routing, synthetic variant producer, Firehose delivery to S3, Data Analytics SQL for pathogenic variant alerting, Lambda-based SNS notification.

**Key decision made:** Kinesis Data Analytics is expensive (~$80/month per KPU). For most genomics alerting scenarios, a Lambda triggered by an S3 event (when Firehose lands a file) is equally effective and 10–15× cheaper. Reserve Data Analytics for stateful stream processing (session windows, cross-stream joins) that Lambda cannot express.

## The Full End-to-End Pipeline

Here is the complete data flow for a realistic genomics research platform, with each component mapped to its exercise:

```
╔══════════════════════════════════════════════════════════════════════════════╗
║                COMPLETE GENOMICS DATA PLATFORM — FLOW DIAGRAM              ║
╚══════════════════════════════════════════════════════════════════════════════╝

  ┌──────────────┐    ┌──────────────┐    ┌──────────────────────────────────┐
  │ SEQUENCER    │    │ PUBLIC DBs   │    │ CLINICAL EHR / LIMS              │
  │ (NovaSeq,    │    │ (NCBI, SRA,  │    │ (HL7 FHIR events,                │
  │  Nanopore)   │    │  ClinVar,    │    │  variant reclassifications)      │
  └──────┬───────┘    │  TCGA/GDC)   │    └────────────────┬─────────────────┘
         │            └──────┬───────┘                     │
         │ real-time         │ scheduled batch             │ real-time
         ▼                   ▼                             ▼
  ┌──────────────┐    ┌──────────────┐             ┌──────────────┐
  │ Kinesis Data │    │ S3 CopyObject│             │ Kinesis      │
  │ Streams      │    │ / NCBI FTP   │             │ Firehose     │
  │ [Ex 9]       │    │ download     │             │ [Ex 9]       │
  └──────┬───────┘    │ [Ex 1, 3]    │             └──────┬───────┘
         │            └──────┬───────┘                    │ 60s buffer
         ├── analytics ──►  KDA [Ex 9] → SNS alerts       │
         │                                                 │
         └────────────────────────────────────────────────┘
                              │
                              ▼
              ╔═══════════════════════════════╗
              ║   S3 BRONZE ZONE [Ex 1]       ║
              ║   raw VCF, FASTQ, JSON, GZIP  ║
              ║   Hive-partitioned             ║
              ║   Never deleted               ║
              ╚═══════════════════════════════╝
                              │
                    ┌─────────┴──────────┐
                    │  Glue Crawler      │
                    │  [Ex 3]            │
                    │  Schema discovery  │
                    └─────────┬──────────┘
                              │
                    ┌─────────┴──────────┐
                    │  Glue PySpark ELT  │
                    │  [Ex 4]            │
                    │  Filter, normalize │
                    │  JSON → Parquet    │
                    │  [Ex 2: format]    │
                    └─────────┬──────────┘
                              │
              ╔═══════════════════════════════╗
              ║   S3 SILVER ZONE [Ex 1,2,4]  ║
              ║   Parquet, partitioned        ║
              ║   by chrom [Ex 7]             ║
              ║   Iceberg for clinical data   ║
              ║   [Ex 6: time travel]         ║
              ╚═══════════════════════════════╝
                              │
                    ┌─────────┴──────────┐
                    │  Athena CTAS +     │
                    │  dbt models [Ex 5] │
                    │  Star schema       │
                    └─────────┬──────────┘
                              │
              ╔═══════════════════════════════╗
              ║   S3 GOLD ZONE [Ex 1,5]      ║
              ║   Fact + dimension tables     ║
              ║   Aggregated, query-optimized ║
              ╚═══════════════════════════════╝
                              │
                    ┌─────────┴──────────┐
                    │  Glue Data Catalog │
                    │  [Ex 3]            │
                    │  Single metadata   │
                    │  registry          │
                    └─────────┬──────────┘
                              │
                    ┌─────────┴──────────┐
                    │  Lake Formation    │
                    │  [Ex 8]            │
                    │  Column + row ACL  │
                    │  HIPAA compliance  │
                    └─────────┬──────────┘
                              │
              ┌───────────────┼──────────────────┐
              ▼               ▼                  ▼
        Amazon Athena    SageMaker         QuickSight
        (SQL queries)    (notebooks)       (dashboards)
        Clinical role    Researcher        Operations
        [Ex 8]           role [Ex 8]       team
```

## AWS Service Reference Card

| Service | Role in the platform | Key pricing unit | When NOT to use it |
|---|---|---|---|
| **S3** | Durable object storage for all three zones | $0.023/GB/month (standard) | When you need sub-millisecond latency (use RDS or DynamoDB) |
| **Athena** | Serverless SQL query engine over S3 | $5.00/TB scanned | For row-by-row updates, high-frequency transactional queries |
| **Glue Data Catalog** | Centralized schema registry | Free up to 1M objects; $1/100K objects beyond | Never avoid — it's cheap and required by Athena + LF |
| **Glue ETL** | Managed PySpark for Bronze→Silver→Gold | $0.44/DPU-hour | For simple transformations < 1 GB/day (use Lambda instead) |
| **Glue Crawlers** | Automated schema discovery | $0.44/DPU-hour | For data you write yourself (use `awswrangler` to register directly) |
| **Kinesis Data Streams** | Low-latency event streaming | $0.015/shard-hour | If you only need to land data to S3 (use Firehose) |
| **Kinesis Firehose** | Managed S3/Redshift/ES delivery | $0.029/GB ingested | If you need sub-60-second S3 landing (use Lambda + S3 PutObject) |
| **Kinesis Data Analytics** | Real-time SQL / Flink on streams | $0.11/KPU-hour | For simple alerting (use Lambda + S3 event triggers instead) |
| **Apache Iceberg** | ACID tables, time travel, schema evolution | No extra cost (Athena billing) | For append-only data (plain Parquet is simpler and faster) |
| **Lake Formation** | Fine-grained column/row access control | No extra cost | When S3 bucket policies are sufficient (single-team, no PHI) |
| **dbt** | SQL transformation pipeline with lineage | Open source (free) | For streaming transformations (dbt is batch-only) |

## Key Architectural Decisions — Cheat Sheet

These are the decisions you will face repeatedly when designing genomics data platforms. Each one has a clear answer given what you have learned.

### Decision 1: Which file format for each zone?

```
Bronze:  Original format  (VCF, FASTQ, BAM, JSON, CSV)
         Never transcode — preserve provenance

Silver:  Parquet  (columnar, compressed, partitioned)
         Exception: use Iceberg Parquet for tables requiring ACID/time travel

Gold:    Parquet  (same as Silver, more aggressively partitioned / pre-aggregated)
```

### Decision 2: When to use Iceberg vs. plain Parquet?

| Use Iceberg when... | Use plain Parquet when... |
|---|---|
| Table needs UPDATE or DELETE | Table is append-only |
| Audit trail / time travel required | No regulatory requirement for history |
| Schema will evolve (new columns) | Schema is stable |
| HIPAA, SOC2, or GDPR applies | Internal research only |

### Decision 3: How many partition keys?

```
Rule: 1–2 partition keys per table, maximum

Good genomics partition keys (cardinality 1–100):  chromosome, tissue_type, year
Bad partition keys (too high cardinality):          sample_id, position, variant_id

Target partition size: 128 MB – 1 GB per partition file
Target partition count: < 10,000 per table
```

### Decision 4: Batch vs. streaming?

```
Use batch (Glue ETL, S3 CopyObject) when:
  • Latency > 15 minutes is acceptable
  • Data arrives as complete files (FASTQ, VCF)
  • Cost is the primary constraint

Use streaming (Kinesis) when:
  • Need real-time alerts (Pathogenic variant detected)
  • Instrument telemetry or sensor data
  • EHR integration with <1 minute SLA
  • Latency < 60 seconds required
```

### Decision 5: Lake Formation vs. S3 bucket policies?

```
S3 bucket policies enforce: bucket/prefix access (all-or-nothing per prefix)
Lake Formation enforces:    table/column/row access (fine-grained)

Use LF when:
  • PHI or dbGaP data is present
  • Different researchers need different column sets
  • Row-level filtering required (e.g., tissue type = 'Brain' only)
  • HIPAA/GDPR compliance must be documented

S3 bucket policies alone are sufficient when:
  • All data is public or de-identified
  • Single team with identical access needs
  • No column-level sensitivity
```

## Cost Sizing: A Realistic Genomics Research Lab

Let's estimate the monthly cost of running this platform for a mid-size genomics lab:

**Scale:** 200 WGS samples/month, 30× coverage, ~100 GB VCF + metadata per sample, 3-year retention

### Storage (S3)

```
Bronze (original VCFs, JSON):
  200 samples/month × 100 GB × 36 months = 720 TB total
  S3 Standard (hot, <6 months): 120 TB × $0.023/GB = $2,856/month
  S3 Glacier Instant (cold, >6 months): 600 TB × $0.004/GB = $2,400/month

Silver (Parquet, ~5x compression):
  720 TB / 5 = 144 TB × $0.023/GB = $3,312/month

Gold (aggregated, ~10x compression from raw):
  72 TB × $0.023/GB = $1,656/month

Total storage: ~$10,224/month
```

### Compute

```
Glue ETL (Bronze→Silver, runs nightly):
  10 DPUs × 2 hours/night × 30 nights × $0.44/DPU-hour = $264/month

Athena queries (analysts running ~500 queries/day):
  500 queries × 5 GB average scan × 30 days × $5.00/TB = $375/month
  (Parquet compression means actual bytes billed ≈ 500 MB per query)
  Realistic after Parquet: $375 / 10 = $37.50/month

Kinesis Firehose (instrument telemetry, 50 GB/month):
  50 GB × $0.029/GB = $1.45/month

Total compute: ~$303/month
```

### Governance and Data Transfer

```
Lake Formation: free (no per-query charge)
Glue Data Catalog: free (< 1M tables)
CloudTrail (compliance logging): ~$10/month
Data transfer out (researcher downloads): 1 TB × $0.09/GB = $92/month

Total governance: ~$102/month
```

### Total Estimated Monthly Cost

| Category | Monthly |
|---|---|
| S3 storage (all zones) | $10,224 |
| Glue ETL | $264 |
| Athena queries | $38 |
| Kinesis Firehose | $1 |
| Governance / logging | $102 |
| **Total** | **~$10,629/month** |

> **Storage dominates the cost** for genomics workloads — this is always true at scale. Compute is cheap relative to the cost of storing petabytes of FASTQ and VCF files. The right lifecycle policy (Bronze → Glacier after 6 months) can cut storage costs by 4× without changing the query interface at all.

## Connecting to Real-World Standards

The architecture you built in these exercises maps directly onto published genomics data standards.

### FAIR Data Principles (Findable, Accessible, Interoperable, Reusable)

| FAIR Principle | How this platform implements it |
|---|---|
| **Findable** | Glue Data Catalog (Ex 3) provides a queryable metadata registry. Crawlers auto-register new datasets. |
| **Accessible** | Athena SQL provides standard access without data movement. Lake Formation (Ex 8) controls who can access what. |
| **Interoperable** | Parquet + Hive partitioning (Ex 2, 7) is readable by Spark, Athena, Trino, Dask, and any Arrow-compatible tool. |
| **Reusable** | Bronze zone preserves original format (Ex 1). Iceberg snapshots (Ex 6) make historical states reproducible. |

### GA4GH Data Connect / DRS

The GA4GH **Data Repository Service (DRS)** is a standard API for resolving data object identifiers to storage URIs. In an AWS implementation:
- DRS IDs map to S3 object keys in the Bronze zone
- The Glue Data Catalog serves as the DRS index
- Lake Formation enforces the access control layer that DRS expects

### dbGaP Data Access Agreements

dbGaP controlled-access datasets (e.g., 1000 Genomes, TCGA controlled tier) require institutional data access agreements. In AWS:
- The IAM researcher role from Exercise 8 maps to a dbGaP data access tier
- Lake Formation column filters implement the "no re-identification" clause
- CloudTrail provides the access audit log that dbGaP's data use monitoring requires
- S3 Object Lock prevents deletion of audit records (WORM compliance)

### HIPAA Safe Harbor vs. Expert Determination

The PHI dataset from Exercise 8 would require one of two de-identification standards before it could be shared:

| Method | What it requires | How the platform helps |
|---|---|---|
| **Safe Harbor** | Remove 18 specific identifiers (name, DOB, zip, MRN, etc.) | Lake Formation column exclusion list matches the 18 identifiers exactly |
| **Expert Determination** | Statistical proof that re-identification risk < 0.04% | Row-level filters (tissue type, cancer type) reduce quasi-identifier combinations |

The researcher IAM role from Exercise 8 — which excluded `patient_name`, `mrn`, `dob`, `ssn_last4` — implements a Safe Harbor-compatible view of the data without copying or modifying the underlying dataset.

## Putting It All Together: Sample End-to-End Workflow

Here is a concrete end-to-end walkthrough for a typical genomics research scenario, annotating which exercise each step comes from.

**Scenario:** A cancer research institute wants to analyze BRCA1/BRCA2 variant frequency across 500 tumor samples, comparing tumor-normal pairs, while ensuring that clinical researchers see full PHI and computational bioinformaticians see only de-identified data.

---

**Step 1 — Data Ingestion (Ex 1, Ex 9)**

Sequencers emit FASTQ files to an S3 event-triggered pipeline. Aligned BAMs and VCF calls from GATK land in `bronze/variants/source=gatk/sample={id}/reference=hg38/`. Simultaneously, a Kinesis Firehose stream captures real-time variant events as the pipeline runs, enabling the clinical team to see preliminary BRCA results within minutes rather than waiting for the full VCF.

---

**Step 2 — Catalog and Format Conversion (Ex 2, Ex 3)**

A Glue Crawler runs hourly over `bronze/` and registers any new VCF-derived CSV tables in the Glue Data Catalog. A nightly Glue ETL job reads these Bronze tables, converts them to Parquet with chromosome partitioning, and writes to `silver/variants/chrom={chr}/`. Parquet compression reduces storage from 50 GB/sample to ~8 GB/sample.

---

**Step 3 — Normalization and QC (Ex 4)**

A PySpark Glue job applies the Silver → Gold transformation: filters low-quality variants (QUAL < 30, depth < 10), annotates with ClinVar classifications, flags BRCA1/BRCA2 hits, and writes an aggregated per-sample BRCA summary to the Gold zone. Running this in Spark across 500 samples takes 20 minutes on 10 DPUs ($1.47) versus hours in a VM.

---

**Step 4 — Dimensional Modeling (Ex 5)**

A dbt model creates `fact_variant_call` (one row per somatic variant per sample) joined to `dim_sample` (tumor stage, tissue, sex, age group), `dim_study` (IRB protocol, PI, cohort), and `dim_date` (sequencing date). The star schema allows analysts to answer "Which tumor stages have the highest BRCA1 pathogenic variant frequency?" with a 3-table join rather than scanning raw VCFs.

---

**Step 5 — Clinical Reclassification Tracking (Ex 6)**

The clinical variant table is stored as an Iceberg table. When a variant is reclassified from VUS to Pathogenic (a common BRCA event), the UPDATE is recorded as a new Iceberg snapshot. The clinical team can query the table as it existed before any reclassification using `FOR TIMESTAMP AS OF`, providing a complete audit trail for the CAP/CLIA accreditation review.

---

**Step 6 — Partitioning Optimization (Ex 7)**

After loading 500 samples, the DBA runs a partition analysis and finds that BRCA-focused queries always filter `WHERE chrom IN ('chr13', 'chr17')`. Re-partitioning the Silver table by chromosome reduces query cost from $0.50/query to $0.01/query for targeted BRCA analyses. The compaction job merges the small streaming files from Exercise 9 into 256 MB Parquet files to reduce per-file S3 overhead.

---

**Step 7 — Access Control (Ex 8)**

Lake Formation enforces three access tiers:
- **Clinical role** (`arn:aws:iam::...:role/clinical`): all columns, all rows, Iceberg time travel enabled
- **Researcher role** (`arn:aws:iam::...:role/researcher`): PHI columns excluded, only samples with IRB consent flag = TRUE
- **Public role** (`arn:aws:iam::...:role/public`): only aggregate Gold tables, no individual-level data

All three roles query the same Athena tables. Lake Formation intercepts each query and rewrites it with the appropriate column/row filter before scanning S3 — no data copying required.

## What This Platform Cannot Do (And What To Use Instead)

No architecture is universal. Know the boundaries:

| Need | Why this platform falls short | Better option |
|---|---|---|
| **Sub-100ms query latency** | Athena cold-start + S3 GET latency is ~1–5 seconds minimum | Amazon Redshift (columnar warehouse) or ElasticSearch |
| **Full-text search over clinical notes** | Athena SQL has no native NLP/text search | Amazon OpenSearch Service |
| **Graph queries** (protein interactions, pathway analysis) | Athena/SQL is relational; graph traversal is O(n²) in SQL | Amazon Neptune (RDF/graph database) |
| **Real-time variant calling** | The platform stores called variants; calling requires GATK/DeepVariant running on aligned reads | AWS Batch + Nextflow / AWS HealthOmics |
| **ML model training** | Athena returns DataFrames; training large models on S3 data is inefficient | SageMaker Training Jobs reading directly from S3 |
| **Transactional EHR writes** | Lake Formation + Athena is read-optimized; concurrent writes at >100 TPS will conflict | Amazon RDS (PostgreSQL) or DynamoDB |
| **BAM/CRAM sequence queries** (e.g., pileup at a position) | Athena cannot parse BAM binary format | AWS HealthOmics Read Store, or htslib on EC2/Batch |

## Concept Map: How the AWS Services Are Related

```
                         ┌──────────────────────┐
                         │   Glue Data Catalog  │
                         │   (metadata registry)│
                         └──────────┬───────────┘
                                    │  schema metadata
                   ┌────────────────┼────────────────┐
                   │                │                │
            ┌──────▼──────┐  ┌──────▼──────┐ ┌──────▼──────┐
            │    Athena   │  │  Glue ETL   │ │    Lake     │
            │  (query)    │  │  (PySpark   │ │  Formation  │
            │  serverless │  │   jobs)     │ │  (access    │
            └──────┬──────┘  └──────┬──────┘ │   control)  │
                   │                │        └──────┬──────┘
                   │ reads          │ reads/         │ governs
                   │                │ writes         │ Athena
                   └────────────────┼────────────────┘
                                    │
                         ┌──────────▼───────────┐
                         │         S3           │
                         │   Bronze / Silver /  │
                         │       Gold           │
                         └──────────────────────┘
                                    ▲
                    ┌───────────────┼───────────────┐
                    │               │               │
             ┌──────┴──────┐ ┌──────┴──────┐ ┌─────┴───────┐
             │  Kinesis    │ │  Kinesis    │ │  Iceberg    │
             │  Firehose   │ │  Streams    │ │  (ACID on   │
             │  (delivery) │ │  (pub/sub)  │ │   Parquet)  │
             └─────────────┘ └──────┬──────┘ └─────────────┘
                                    │
                             ┌──────▼──────┐
                             │  Kinesis    │
                             │  Analytics  │
                             │  (real-time │
                             │   SQL/Flink)│
                             └─────────────┘
```

**The Glue Data Catalog is the hub.** Every other service reads or writes to it — Athena uses it to resolve table locations, Glue ETL uses it to discover input schemas, Lake Formation uses it as the authorization namespace. You can think of the Data Catalog as the "registry" layer that makes S3 feel like a database.

## Common Mistakes and How to Avoid Them

These are the most common errors students (and professionals) make when building genomics data lakes on AWS:

### 1. Storing everything in the same S3 prefix with no partitioning
**Symptom:** Athena scans the entire dataset for every query regardless of filter.  
**Fix:** Add Hive-style partition keys (`chrom=chr17/`) and register them in the Glue catalog. (Exercise 7)

### 2. Skipping the Bronze zone and transforming on ingest
**Symptom:** A bug in the transformation corrupts data and you have no way to recover.  
**Fix:** Always write raw data to Bronze first, then transform to Silver. (Exercise 1)

### 3. Using S3 bucket policies as the only access control
**Symptom:** A researcher can see PHI columns they should not have access to.  
**Fix:** Enable Lake Formation and grant column-level permissions. S3 bucket policies are coarse-grained. (Exercise 8)

### 4. Over-partitioning into millions of tiny files
**Symptom:** Athena queries are slow and expensive even on small datasets; Glue catalog has millions of partitions.  
**Fix:** Target 128 MB–1 GB per partition file. Compact small files with `OPTIMIZE` (Iceberg) or a Glue compaction job. (Exercise 7)

### 5. Using Kinesis Data Analytics when Lambda is sufficient
**Symptom:** $80+/month for a streaming pipeline that fires at most a few alerts per day.  
**Fix:** Use Firehose → S3 event → Lambda for simple alerting. Reserve KDA for stateful stream processing. (Exercise 9)

### 6. Running Athena queries without specifying a partition filter
**Symptom:** A query like `SELECT * FROM silver_variants WHERE gene = 'BRCA1'` scans the entire 10 TB Silver table.  
**Fix:** Always include the partition column (e.g., `WHERE chrom = 'chr17' AND gene = 'BRCA1'`) to trigger partition pruning. (Exercises 1, 7)

### 7. Using plain Parquet for tables that need audit trails
**Symptom:** After a variant reclassification, you cannot reconstruct what the table looked like 6 months ago.  
**Fix:** Use Apache Iceberg for any table with UPDATE/DELETE requirements or regulatory audit obligations. (Exercise 6)

### 8. Forgetting to delete Kinesis shards and Glue crawlers after exercises
**Symptom:** Unexpected AWS bill from idle resources.  
**Fix:** Run the cleanup notebook (Exercise 0) after every session. Kinesis charges by the shard-hour even when empty.

## The Data Lifecycle at a Glance

One of the most important concepts in this course is that data has a lifecycle — it moves through zones as it ages and as our understanding of it matures.

```
Time ──────────────────────────────────────────────────────────────►

Day 1           Week 1          Month 1         Year 1+
────────        ────────        ────────        ────────
Sequencer  →    Glue ETL   →    Athena    →     Glacier
emits FASTQ     converts to     queries         (cold storage,
                Parquet         drive           $0.004/GB)
                Bronze →        clinical
                Silver →        decisions
                Gold

S3 Standard     S3 Standard     S3 Standard     S3 Glacier
$0.023/GB       $0.023/GB       $0.023/GB       $0.004/GB

Streaming        Batch ETL      Ad hoc SQL      Archival
(Kinesis)        (Glue)         (Athena)        (restore on
                                                 demand)
```

### S3 Lifecycle Policies

AWS S3 lifecycle policies automate the transition between storage tiers:

```json
{
  "Rules": [
    {
      "Prefix": "bronze/",
      "Transitions": [
        { "Days": 90,  "StorageClass": "STANDARD_IA" },
        { "Days": 180, "StorageClass": "GLACIER_IR" },
        { "Days": 365, "StorageClass": "DEEP_ARCHIVE" }
      ]
    },
    {
      "Prefix": "silver/",
      "Transitions": [
        { "Days": 180, "StorageClass": "STANDARD_IA" }
      ]
    }
  ]
}
```

For a genomics lab keeping BAMs and FASTQs for 5+ years, transitioning Bronze to Deep Archive after 1 year reduces storage cost from $0.023/GB to $0.00099/GB — a **23× reduction** on the oldest files, which typically represent the bulk of stored bytes.

## Capstone Exercise: Design Review

Apply everything you have learned to answer the following design questions. These are the kinds of questions you would face in a real architecture review or a technical interview for a bioinformatics data engineering role.

---

**Question 1 — Architecture Design**

A biobank wants to build a data platform for 1 million whole-genome sequences over 10 years. Each genome produces:
- 200 GB of raw FASTQ files
- 50 GB aligned BAM
- 5 GB gVCF (all positions called)
- 100 MB small variant VCF
- 10 MB sample metadata JSON

Sketch the S3 zone structure, partition keys for each file type, Glue ETL schedule, and estimated monthly storage cost. What lifecycle policy would you apply to FASTQs vs. VCFs?

---

**Question 2 — Governance Design**

The biobank has three user groups:
- **Internal clinical lab:** needs full data access including PHI
- **Academic collaborators (NIH-funded):** need variant data, no PHI, restricted to consented samples only
- **Commercial partners:** need summary statistics only, no individual-level data

Design the Lake Formation permission model. What IAM roles, column filters, and row-level data cell filters would you create? How would you audit access for the NIH data use reporting requirement?

---

**Question 3 — Performance Optimization**

Analysts complain that their most common query — "count pathogenic variants in BRCA1/BRCA2 per tumor type, excluding low-quality calls" — takes 4 minutes and costs $8.00 per run. The Silver variants table is 50 TB, plain Parquet, partitioned only by `year`. Describe at least three changes you would make to reduce this query to under 10 seconds and under $0.10.

---

**Question 4 — Streaming vs. Batch**

An oncologist requests that she be notified within 5 minutes whenever a newly sequenced tumor sample has a Pathogenic variant in any actionable cancer gene (BRCA1, BRCA2, TP53, EGFR, KRAS). Currently, variant calls complete and land in S3 as VCF files within 3 hours of the sequencer run completing. Design the streaming alerting pipeline. Which Kinesis services would you use, and why? What is the estimated monthly cost if 50 samples are sequenced per day?

---

**Question 5 — HIPAA Compliance**

List the specific AWS configuration steps required to make this platform HIPAA-eligible. Your answer should cover: encryption at rest, encryption in transit, access logging, audit trails, Business Associate Agreement (BAA), and minimum-necessary access controls. Which of these does Lake Formation handle, and which require separate configuration?

## Answers Space

*(Double-click here to write your answers to the capstone questions)*

---

**Question 1 — Architecture Design**



---

**Question 2 — Governance Design**



---

**Question 3 — Performance Optimization**



---

**Question 4 — Streaming vs. Batch**



---

**Question 5 — HIPAA Compliance**




## Further Reading and Resources

### AWS Documentation
- [Amazon S3 Storage Classes](https://aws.amazon.com/s3/storage-classes/) — storage tier comparison and lifecycle pricing
- [Amazon Athena Best Practices](https://docs.aws.amazon.com/athena/latest/ug/performance-tuning.html) — partitioning, compression, and cost optimization
- [AWS Lake Formation Developer Guide](https://docs.aws.amazon.com/lake-formation/latest/dg/what-is-lake-formation.html) — column/row security in depth
- [Apache Iceberg on AWS](https://docs.aws.amazon.com/athena/latest/ug/querying-iceberg.html) — Athena Iceberg operations reference
- [Kinesis Data Streams Developer Guide](https://docs.aws.amazon.com/streams/latest/dev/introduction.html) — shard management, enhanced fan-out

### Genomics-Specific Resources
- [AWS Genomics CLI (AGC)](https://aws.github.io/amazon-genomics-cli/) — workflow orchestration for genomics on AWS
- [AWS HealthOmics](https://aws.amazon.com/omics/) — purpose-built for genomics storage and analytics (Sequence Store, Variant Store)
- [GATK Best Practices](https://gatk.broadinstitute.org/hc/en-us/sections/360007226651) — variant calling workflows that produce the data this lake ingests
- [GA4GH Data Connect](https://ga4gh-discovery.github.io/data-connect/) — federated table API standard implemented over lakes like this one

### Data Engineering Concepts
- [The Data Lakehouse Paper (Zaharia et al., 2021)](https://www.cidrdb.org/cidr2021/papers/cidr2021_paper17.pdf) — the academic foundation for Delta Lake / Iceberg / Hudi
- [dbt Documentation](https://docs.getdbt.com/) — transformation framework used in Exercise 5
- [Apache Iceberg Specification](https://iceberg.apache.org/spec/) — how table metadata, snapshots, and manifests work
- [FAIR Data Principles](https://www.go-fair.org/fair-principles/) — the framework referenced throughout Exercise 10

### Cost and Optimization Tools
- [AWS Pricing Calculator](https://calculator.aws/) — estimate monthly costs before deploying
- [AWS Cost Explorer](https://aws.amazon.com/aws-cost-management/aws-cost-explorer/) — analyze actual spend after deployment
- [S3 Storage Lens](https://aws.amazon.com/s3/features/storage-lens/) — identify small files, unused buckets, and lifecycle opportunities

---

## Congratulations

You have worked through the complete BINFX410 Chapter 10 exercise series. The platform you designed here — S3 medallion architecture, Glue ETL, Athena SQL, Iceberg ACID tables, Lake Formation governance, and Kinesis streaming — is the same architecture used in production at leading genomics institutions, biobanks, and clinical labs worldwide.

The core insight to carry forward:

> **A genomics data lake is not a technology — it is a set of organizational conventions (zones, formats, partitions, catalog, governance) layered on top of cheap object storage and serverless compute. The conventions are what make petabytes of variant data queryable, reproducible, and governed. The technology just enforces them.**